# Inference - configurable

In [1]:
import os, glob
import numpy as np
import pandas as pd
import cv2

IMG_H, IMG_W = 128, 800
DEFECTS = ["defect_a", "defect_b", "defect_c", "defect_d"]
BS = 16

CONFIG = dict(models=["scratch"], tta={"scratch": True}, tau=0.5,
              min_area=[0, 500, 200, 300], gate=[0.98, 0.98, 0.9, 0.98])

def find_file(name, known=None):
    if known and os.path.exists(known):
        return known
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name} not found - attach the right input to this notebook")
    return hits[0]

def find_image_dir(sample_id):
    hits = glob.glob(f"/kaggle/input/**/{sample_id}.*", recursive=True)
    if not hits:
        raise FileNotFoundError(f"image {sample_id} not found - attach the competition data")
    return os.path.dirname(hits[0])

SAMPLE_CSV = find_file("sample_submission.csv", "/kaggle/input/competitions/industrial-surface-defect-detection/public/sample_submission.csv")
CKPTS = {}
if "scratch" in CONFIG["models"]:
    CKPTS["scratch"] = find_file("best_unet_scratch.pt", "/kaggle/input/notebooks/mruniksurya/milestone-2/best_unet_scratch.pt")
if "q8" in CONFIG["models"]:
    CKPTS["q8"] = find_file("best_unet_q8.pt", "/kaggle/input/notebooks/mruniksurya/milestone-2-q8/best_unet_q8.pt")
print("CONFIG:", CONFIG); print("checkpoints:", CKPTS)

CONFIG: {'models': ['scratch'], 'tta': {'scratch': True}, 'tau': 0.5, 'min_area': [0, 500, 200, 300], 'gate': [0.98, 0.98, 0.9, 0.98]}
checkpoints: {'scratch': '/kaggle/input/notebooks/mruniksurya/milestone-2/best_unet_scratch.pt'}


## 1. RLE encoder and post-processing

In [2]:
def rle_decode(rle, shape=(IMG_H, IMG_W)):
    """1-based, column-major RLE string -> binary (H, W) mask. NaN / blank -> all zeros."""
    flat = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    if isinstance(rle, str) and rle.strip():
        tok = np.array(rle.split(), dtype=np.int64)
        starts = tok[0::2] - 1
        ends = starts + tok[1::2]
        for s, e in zip(starts, ends):
            flat[max(s, 0):min(e, flat.size)] = 1
    return flat.reshape(shape, order="F")

def rle_encode(mask):
    """Binary (H, W) mask -> RLE string (1-based, column-major). Empty mask -> empty string."""
    flat = np.asarray(mask).flatten(order="F")
    padded = np.concatenate([[0], flat, [0]])
    k = np.where(padded[1:] != padded[:-1])[0] + 1
    return " ".join(f"{s} {e - s}" for s, e in zip(k[0::2], k[1::2]))

In [3]:
import cv2

def postprocess(prob, tau, min_area, gate):
    """prob (H,W) of ONE (image, class) pair -> binary uint8 mask.
    gate: if the most confident pixel is below it, the whole mask is emptied.
    min_area: connected regions (8-connectivity) smaller than this many pixels are removed."""
    if prob.max() < gate:
        return np.zeros(prob.shape, dtype=np.uint8)
    mask = (prob > tau).astype(np.uint8)
    if min_area > 0 and mask.any():
        n, lab, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
        keep = np.zeros(n, dtype=bool)
        keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= min_area
        mask = keep[lab].astype(np.uint8)
    return mask

## 2. Models and preprocessing (identical to the training notebooks and the analysis notebook)

In [ ]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    """(Conv3x3 -> BatchNorm -> ReLU) x 2 : the basic U-Net building block."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.block(x)

class UNetScratch(nn.Module):
    """Milestone-2 main model: 4 encoder levels, 1 input channel, 4 logits. 7,762,564 parameters."""
    def __init__(self, in_ch=1, n_classes=4, base=32):
        super().__init__()
        c = [base, base * 2, base * 4, base * 8, base * 16]
        self.enc1, self.enc2 = DoubleConv(in_ch, c[0]), DoubleConv(c[0], c[1])
        self.enc3, self.enc4 = DoubleConv(c[1], c[2]), DoubleConv(c[2], c[3])
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(c[3], c[4])
        self.up4, self.dec4 = nn.ConvTranspose2d(c[4], c[3], 2, stride=2), DoubleConv(c[3] * 2, c[3])
        self.up3, self.dec3 = nn.ConvTranspose2d(c[3], c[2], 2, stride=2), DoubleConv(c[2] * 2, c[2])
        self.up2, self.dec2 = nn.ConvTranspose2d(c[2], c[1], 2, stride=2), DoubleConv(c[1] * 2, c[1])
        self.up1, self.dec1 = nn.ConvTranspose2d(c[1], c[0], 2, stride=2), DoubleConv(c[0] * 2, c[0])
        self.head = nn.Conv2d(c[0], n_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)

class UNet3Stage(nn.Module):
    """Q8 model: 3 down stages, 3 input channels (gray replicated), 4 logits. 1,927,108 parameters."""
    def __init__(self, in_ch=3, out_ch=4, base=32):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base)
        self.enc2 = DoubleConv(base, base * 2)
        self.enc3 = DoubleConv(base * 2, base * 4)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base * 4, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = DoubleConv(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)
        self.head = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

IMG_MEAN, IMG_STD = 0.3314, 0.1892      # training-set statistics used by the scratch model (milestone-2 notebook)

def to_input(u8, kind):
    """u8: (B,H,W) uint8 grayscale -> model input tensor.
    scratch: (pixel/255 - mean)/std, 1 channel.   q8: pixel/255 only, gray replicated to 3 channels (what cv2 BGR->RGB gave)."""
    x = torch.from_numpy(np.ascontiguousarray(u8)).float().div_(255.0)
    if kind == "scratch":
        return ((x - IMG_MEAN) / IMG_STD).unsqueeze(1)
    return x.unsqueeze(1).repeat(1, 3, 1, 1)

@torch.no_grad()
def predict_probs(model, kind, u8, tta=False):
    """Sigmoid probabilities (B,4,H,W) on the GPU. tta=True averages the plain image with its horizontal and vertical flip."""
    x = to_input(u8, kind).to(DEVICE)
    p = torch.sigmoid(model(x))
    if tta:
        p = p + torch.sigmoid(model(x.flip(-1))).flip(-1)      # horizontal flip, un-flip the output
        p = p + torch.sigmoid(model(x.flip(-2))).flip(-2)      # vertical flip, un-flip the output
        p = p / 3.0
    return p

def load_model(kind, ckpt):
    m = UNetScratch(in_ch=1, n_classes=4, base=32) if kind == "scratch" else UNet3Stage(in_ch=3, out_ch=4, base=32)
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return m.to(DEVICE).eval()

In [4]:
models = {kind: load_model(kind, path) for kind, path in CKPTS.items()}
for kind, m in models.items():
    print(kind, f"{count_params(m):,} parameters")

NameError: name 'load_model' is not defined

## 4. Predict the test set

In [ ]:
sample = pd.read_csv(SAMPLE_CSV)
test_ids = sample["image_id"].drop_duplicates().tolist()
img_dir = find_image_dir(test_ids[0])
files = {os.path.splitext(f)[0]: f for f in os.listdir(img_dir)}
print("test images:", len(test_ids), "| folder:", img_dir)

rle_lookup = {}
for s in range(0, len(test_ids), BS):
    ids = test_ids[s:s + BS]
    u8 = np.stack([cv2.imread(os.path.join(img_dir, files[i]), cv2.IMREAD_GRAYSCALE) for i in ids])
    assert u8.shape[1:] == (IMG_H, IMG_W), u8.shape
    p = None
    for kind in CONFIG["models"]:
        q = predict_probs(models[kind], kind, u8, CONFIG["tta"].get(kind, False))
        p = q if p is None else p + q
    probs = (p / len(CONFIG["models"])).float().cpu().numpy()            # (B,4,H,W)
    for b, iid in enumerate(ids):
        for k, d in enumerate(DEFECTS):                                  # channel order a,b,c,d as in training
            m = postprocess(probs[b, k], CONFIG["tau"], CONFIG["min_area"][k], CONFIG["gate"][k])
            r = rle_encode(m)
            rle_lookup[(iid, d)] = r if r else " "
print("predicted pairs:", len(rle_lookup))

## 5. submission and sanity-check

In [ ]:
sub = sample.copy()
sub["mask_rle"] = [rle_lookup[(i, d)] for i, d in zip(sub["image_id"], sub["defect_type"])]
sub.to_csv("/kaggle/working/submission.csv", index=False)

assert len(sub) == len(sample) == 12000 and list(sub.columns) == list(sample.columns)
empty = sub["mask_rle"].str.strip().eq("")
print("rows:", len(sub), "| overall empty rate:", round(float(empty.mean()), 4))
print("empty rate per class:"); print(empty.groupby(sub["defect_type"]).mean().round(3).to_string())
print("reference: scratch raw tau 0.8 gave overall 0.4803 (a 0.551, b 0.606, c 0.456, d 0.308); train true empty rates are about a 0.905, b 0.973, c 0.64, d 0.85")
sub.head()